# GroupDNA - WhatsApp Chat Analyzer(hostel boys)

**Name:** Navyashree S


Import Libraries

In [ ]:
# NumPy is used for arrays and simple numerical calculations.
import numpy as np
from datetime import datetime


Reading  the Dataset

In [ ]:
with open("hostel_bois.txt", "r", encoding="utf-8") as file:
    lines = file.readlines()

print("Total Lines:", len(lines))


Total Lines: 3178


Initialize Variables

In [ ]:
# These lists/counters store parsed messages and special chat entries.
messages = []

system_total = 0
media_total = 0
deleted_total = 0


Parse the Chat

In [ ]:
# Read each chat line and separate normal messages from special entries.
# These lists/counters store parsed messages and special chat entries.
system_messages = []

for line in lines:
    line = line.strip()

    if line == "" or " - " not in line:
        continue

    timestamp, rest = line.split(" - ", 1)

    # System entries do not contain a sender
    if ": " not in rest:
        system_count += 1
        system_messages.append(line)
        continue

    sender, text = rest.split(": ", 1)

    # Count special WhatsApp messages separately
    if text == "<Media omitted>":
        media_count += 1
        continue

    if text == "This message was deleted":
        deleted_count += 1
        continue

    messages.append({
        "timestamp": timestamp,
        "sender": sender,
        "text": text
    })

print("System Messages:", system_count)
print("Media Messages:", media_count)
print("Deleted Messages:", deleted_count)


System Messages: 4
Media Messages: 32
Deleted Messages: 15


In [ ]:
print("=" * 40)
print("PARSER SUMMARY")
print("=" * 40)
print("Total Lines      :", len(lines))
print("Valid Messages   :", len(messages))
print("System Messages  :", system_count)
print("Media Messages   :", media_count)
print("Deleted Messages :", deleted_count)


PARSER SUMMARY
Total Lines      : 3178
Valid Messages   : 3127
System Messages  : 4
Media Messages   : 32
Deleted Messages : 15


Verify the Parser

In [ ]:
print("Total Lines:", len(lines))
print("Valid Messages:", len(messages))
print("Special/System Entries:", len(lines) - len(messages))


Total Lines: 3178
Valid Messages: 3127
Special/System Entries: 51


# Feature 2 : Group Overview

Total Messages

In [ ]:
# Count all valid messages after removing special entries.
total_messages = len(messages)

print("Total Messages :", total_messages)


Total Messages : 3127


Total Participants

In [ ]:
participants = set()

for msg in messages:
    participants.add(msg["sender"])

print("Total Participants :", len(participants))
print("Participants :", sorted(participants))


Total Participants : 6
Participants : ['Aman', 'Karan', 'Neha', 'Priya', 'Rahul', 'Vikas']


In [ ]:
message_count = {}

for msg in messages:
    sender = msg["sender"]
    message_count[sender] = message_count.get(sender, 0) + 1


Messages Per Person

In [ ]:
print("=" * 40)
print("Messages Sent by Each Participant")
print("=" * 40)

sorted_messages = sorted(message_count.items(), key=lambda x: x[1], reverse=True)

for person, count in sorted_messages:
    print(f"{person:<10} : {count}")


Messages Sent by Each Participant
Rahul      : 940
Priya      : 712
Neha       : 624
Aman       : 484
Karan      : 345
Vikas      : 22


# Feature 3 : Most Active Day

In [ ]:
# Find when the group was most active.
day_count = {}

for msg in messages:
    date = msg["timestamp"].split(",")[0]
    day_count[date] = day_count.get(date, 0) + 1

most_active_day = max(day_count, key=day_count.get)

print("Most Active Day :", most_active_day)
print("Messages :", day_count[most_active_day])


Most Active Day : 04/05/24
Messages : 74


# Feature 4 : Most Active Hour

In [ ]:
# Find when the group was most active.
hour_count = {}

for msg in messages:
    time = msg["timestamp"].split(",")[1].strip()
    hour = int(time.split(":")[0])
    hour_count[hour] = hour_count.get(hour, 0) + 1

most_active_hour = max(hour_count, key=hour_count.get)

print("Most Active Hour :", most_active_hour)
print("Messages :", hour_count[most_active_hour])


Most Active Hour : 18
Messages : 244


# Feature 5 : Activity Heatmap

Create NumPy Matrix

In [ ]:
# Build a simple participant-by-hour activity matrix using NumPy.
# Make one row for each real participant
# Get the names of people who actually sent messages.
participants = sorted(participants)

person_index = {person: i for i, person in enumerate(participants)}
heatmap = np.zeros((len(participants), 24), dtype=int)


In [ ]:
# Build a simple participant-by-hour activity matrix using NumPy.
for msg in messages:
    sender = msg["sender"]
    hour = int(msg["timestamp"].split(",")[1].strip().split(":")[0])
    heatmap[person_index[sender]][hour] += 1

print("Heatmap Matrix:\n")
print(heatmap)


Heatmap Matrix:

[[ 53  67  66  60  87   0   0   0   0   0   0   0   0   0  14  11  18   5
   16   8  12  11   0  56]
 [  0   0   0   0   0   0   0   4  11  16  19  16  36  22  32  26  27  27
   24  32  23  14   9   7]
 [  0   0   0   0   0  19   3  13  35  51  52  21  39  36  26   9  36  47
   62  49  44  26  26  30]
 [  0   0   0   0   0   0  13  20  46  64  61  61  57  48  44  28  32  40
   38  59  42  32  18   9]
 [  3  15  17  17  22   9  17  17  23  17  25  15  57  48  45  53  71  48
  102  74  40  92  60  53]
 [  0   0   0   0   0   0   0   1   2   1   1   0   1   2   0   1   1   3
    2   2   1   1   1   2]]


In [ ]:
print("=" * 95)
print("ACTIVITY HEATMAP (Hours 00-23)")
print("=" * 95)

print("Participant".ljust(12), end=" ")
for hour in range(24):
    print(f"{hour:02}", end=" ")
print()

print("-" * 95)

# Shade is based on each person's own activity level
for i, person in enumerate(participants):
    print(person.ljust(12), end=" ")

    row_max = max(heatmap[i])

    for value in heatmap[i]:
        if value == 0:
            symbol = "*"
        elif row_max == 0:
            symbol = "░"
        elif value <= row_max * 0.25:
            symbol = "░"
        elif value <= row_max * 0.50:
            symbol = "▒"
        elif value <= row_max * 0.75:
            symbol = "▓"
        else:
            symbol = "█"

        print(symbol, end="  ")

    print()


ACTIVITY HEATMAP (Hours 00-23)
Participant  00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
-----------------------------------------------------------------------------------------------
Aman         ▓  █  █  ▓  █  *  *  *  *  *  *  *  *  *  ░  ░  ░  ░  ░  ░  ░  ░  *  ▓  
Karan        *  *  *  *  *  *  *  ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░  
Neha         *  *  *  *  *  ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒  
Priya        *  *  *  *  *  *  ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░  
Rahul        ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Vikas        *  *  *  *  *  *  *  ▒  ▓  ▒  ▒  *  ▒  ▓  *  ▒  ▒  █  ▓  ▓  ▒  ▒  ▒  ▓  


# Feature 6 : Top Words

In [ ]:
# Remove common words and display the most frequent useful words.
# A few common words are removed from the word count
stop_words = {
    "the", "and", "for", "you", "that", "this", "with", "are", "was",
    "have", "has", "had", "but", "not", "what", "when", "where", "who",
    "how", "from", "they", "will", "just", "been", "your", "our", "its",
    "about", "can", "all", "any", "yes", "yeah", "like"
}

word_count = {}

for msg in messages:
    words = msg["text"].lower().split()

    for word in words:
        word = word.strip(".,!?()[]{}\"'_:;-")

        if len(word) < 2 or word in stop_words:
            continue

        word_count[word] = word_count.get(word, 0) + 1

sorted_words = sorted(word_count.items(), key=lambda x: x[1], reverse=True)

print("=" * 40)
print("Top 10 Words")
print("=" * 40)

for word, count in sorted_words[:10]:
    print(f"{word:<15} {count}")


Top 10 Words
to              667
is              591
in              388
guys            318
so              292
hai             268
am              260
today           257
at              257
my              223


# Feature 7 : Response Speed and Silent Streaks

In [ ]:
# Calculate reply gaps to understand response speed and quiet periods.
# Find reply gaps between two different participants
response_times = {person: [] for person in participants}
silent_gaps = []

parsed_messages = []

for msg in messages:
    dt = datetime.strptime(msg["timestamp"], "%d/%m/%y, %H:%M")
    parsed_messages.append((dt, msg))

for i in range(1, len(parsed_messages)):
    previous_time, previous_msg = parsed_messages[i - 1]
    current_time, current_msg = parsed_messages[i]

    gap = (current_time - previous_time).total_seconds() / 60

    if current_msg["sender"] != previous_msg["sender"]:
        response_times[current_msg["sender"]].append(gap)

    # More than 6 hours without a message = silent streak
    if gap >= 360:
        silent_gaps.append((gap, previous_time, current_time))

print("=" * 40)
print("Average Response Time")
print("=" * 40)

for person in participants:
    times = response_times[person]
    if times:
        print(f"{person:<10} : {sum(times)/len(times):.1f} minutes")
    else:
        print(f"{person:<10} : No reply data")

silent_gaps.sort(reverse=True, key=lambda x: x[0])

print("\nLongest Silent Streaks:")
for gap, start, end in silent_gaps[:5]:
    print(f"{gap/60:.1f} hours  ({start.strftime('%d/%m %H:%M')} - {end.strftime('%d/%m %H:%M')})")


Average Response Time
Aman       : 54.9 minutes
Karan      : 36.8 minutes
Neha       : 41.3 minutes
Priya      : 42.6 minutes
Rahul      : 35.2 minutes
Vikas      : 34.9 minutes

Longest Silent Streaks:


# Feature 8 : Most Active Member

In [ ]:
most_active = max(message_count, key=message_count.get)

print("Most Active Member :", most_active)
print("Messages Sent :", message_count[most_active])


Most Active Member : Rahul
Messages Sent : 940


# Feature 9 : Personality Archetypes

In [ ]:
# Give each participant one simple personality type using message behaviour.
print("=" * 50)
print("PERSONALITY ARCHETYPES")
print("=" * 50)

# Simple scores are used so every person gets one archetype.
stats = {}

care_words = {"eat", "food", "home", "study", "sleep", "take", "care"}
fun_words = {"haha", "hahaha", "lol", "😂", "🤣", "lmao"}
drama_words = {"omg", "seriously", "why", "ugh", "bro", "damn"}

for person in participants:
    person_msgs = [m for m in messages if m["sender"] == person]

    total = len(person_msgs)
    questions = sum("?" in m["text"] for m in person_msgs)
    exclaims = sum("!" in m["text"] for m in person_msgs)
    night = 0
    care = 0
    fun = 0
    drama = 0
    words_total = 0

    for m in person_msgs:
        hour = int(m["timestamp"].split(",")[1].strip().split(":")[0])
        if hour >= 22 or hour <= 5:
            night += 1

        words = m["text"].lower().split()
        words_total += len(words)

        for w in words:
            w = w.strip(".,!?()[]{}\"'_:;-")
            if w in care_words:
                care += 1
            if w in fun_words:
                fun += 1
            if w in drama_words:
                drama += 1

    stats[person] = {
        "messages": total,
        "questions": questions,
        "exclaims": exclaims,
        "night_ratio": night / total if total else 0,
        "avg_words": words_total / total if total else 0,
        "care": care,
        "fun": fun,
        "drama": drama
    }

# Each person gets the archetype with the highest score.
archetype_scores = {}

for person in participants:
    s = stats[person]
    archetype_scores[person] = {
        "The Spammer": s["messages"],
        "The Group Mom": s["care"] * 8 + s["exclaims"] * 0.1,
        "The Night Owl": s["night_ratio"] * 100,
        "The Storyteller": s["avg_words"] * 3,
        "The Drama Queen": s["drama"] * 10 + s["exclaims"],
        "The Ghost": 1 / s["messages"] * 100 if s["messages"] else 100,
        "The Comedian": s["fun"] * 15,
        "The Question Master": s["questions"] * 12
    }

archetypes = {}
for person in participants:
    archetypes[person] = max(
        archetype_scores[person],
        key=archetype_scores[person].get
    )

for person in sorted(participants):
    print(f"{person:<10} : {archetypes[person]}")


PERSONALITY ARCHETYPES
Aman       : The Group Mom
Karan      : The Drama Queen
Neha       : The Drama Queen
Priya      : The Question Master
Rahul      : The Spammer
Vikas      : The Comedian


# Final GroupDNA Report

In [ ]:
# Find when the group was most active.
print("=" * 60)
print("GROUPDNA WHATSAPP CHAT ANALYSIS REPORT")
print("=" * 60)

print(f"Total Messages      : {len(messages)}")
print(f"Participants        : {len(participants)}")
print(f"Most Active Person  : {most_active}")
print(f"Most Active Day     : {most_active_day}")
print(f"Most Active Hour    : {most_active_hour}:00")
print(f"Media Messages      : {media_count}")
print(f"Deleted Messages    : {deleted_count}")

print("\nPersonality Summary:")
for person in sorted(participants):
    print(f"{person:<10} : {archetypes[person]}")

print("=" * 60)
print("Thank you for using GroupDNA WhatsApp Chat Analyzer")
print("=" * 60)


GROUPDNA WHATSAPP CHAT ANALYSIS REPORT
Total Messages      : 3127
Participants        : 6
Most Active Person  : Rahul
Most Active Day     : 04/05/24
Most Active Hour    : 18:00
Media Messages      : 32
Deleted Messages    : 15

Personality Summary:
Aman       : The Group Mom
Karan      : The Drama Queen
Neha       : The Drama Queen
Priya      : The Question Master
Rahul      : The Spammer
Vikas      : The Comedian
Thank you for using GroupDNA WhatsApp Chat Analyzer
